# Pipeline Colab — Poverty SAE Niger

Entraine un ResNet-18 sur les patches Landsat 7 des 476 clusters DHS Niger 2012.

**Validation** : Group 5-fold par departement (generalisation spatiale).

**Duree** : ~2h (T4 GPU)

**Runtime** : Menu -> Runtime -> Change runtime type -> T4 GPU

## 1. Monter Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Configurer les chemins

Le dossier Drive doit contenir : les 476 .tif (export GEE) + `clusters_gps.csv` + `entrainer_resnet.py`.

Modifier `DRIVE_PATH` si necessaire.

In [ ]:
import os, shutil, glob
import numpy as np
import pandas as pd

# A MODIFIER : dossier Drive contenant les .tif + csv + script
DRIVE_PATH = "/content/drive/MyDrive/poverty_sae_patches"

# Working directory sur le disque local de Colab (rapide)
WORKING = "/content/poverty_sae"

print(f"Drive  : {DRIVE_PATH}")
print(f"Local  : {WORKING}")

## 3. Verifier les fichiers sources

In [ ]:
tif_count = len(glob.glob(f"{DRIVE_PATH}/*.tif"))
has_labels = os.path.exists(f"{DRIVE_PATH}/clusters_gps.csv")
has_script = os.path.exists(f"{DRIVE_PATH}/entrainer_resnet.py")

print(f"Patches .tif : {tif_count}")
print(f"clusters_gps.csv : {'OK' if has_labels else 'MANQUANT'}")
print(f"entrainer_resnet.py : {'OK' if has_script else 'MANQUANT'}")

assert tif_count > 0, "Aucun .tif trouve dans " + DRIVE_PATH
assert has_labels, "clusters_gps.csv manquant"
assert has_script, "entrainer_resnet.py manquant"
print("Tout est la. Pret.")

## 4. Copier vers le disque local (rapide)

In [ ]:
os.makedirs(f"{WORKING}/data/processed/patches_landsat", exist_ok=True)
os.makedirs(f"{WORKING}/outputs/tables", exist_ok=True)
os.makedirs(f"{WORKING}/outputs/models", exist_ok=True)
os.makedirs(f"{WORKING}/outputs/figures", exist_ok=True)

# Copier les .tif
for f in glob.glob(f"{DRIVE_PATH}/*.tif"):
    shutil.copy(f, f"{WORKING}/data/processed/patches_landsat/")

# Copier le script et les labels
shutil.copy(f"{DRIVE_PATH}/entrainer_resnet.py", f"{WORKING}/entrainer_resnet.py")
shutil.copy(f"{DRIVE_PATH}/clusters_gps.csv", f"{WORKING}/data/processed/clusters_gps.csv")

n = len(glob.glob(f"{WORKING}/data/processed/patches_landsat/*"))
print(f"Fichiers copies : {n}")

## 5. Convertir .tif -> .npy

In [ ]:
patch_dir = f"{WORKING}/data/processed/patches_landsat"
tif_files = glob.glob(f"{patch_dir}/*.tif")
npy_files = glob.glob(f"{patch_dir}/*.npy")

if len(tif_files) > 0 and len(npy_files) == 0:
    !pip install rasterio -q
    import rasterio
    print(f"Conversion de {len(tif_files)} .tif -> .npy...")
    for tif_path in tif_files:
        basename = os.path.basename(tif_path).replace(".tif", "")
        cluster_id = int(basename.split("_")[1])
        with rasterio.open(tif_path) as src:
            patch = src.read()
        patch = np.transpose(patch, (1, 2, 0)).astype(np.float32)
        patch = np.nan_to_num(patch, nan=0.0)
        p2, p98 = np.percentile(patch, [2, 98])
        patch = np.clip((patch - p2) / (p98 - p2 + 1e-8), 0, 1)
        np.save(os.path.join(patch_dir, f"{cluster_id}.npy"), patch.astype(np.float32))
    print("Conversion terminee")
else:
    print("Patches deja en .npy, rien a faire")

## 6. Verification finale

In [ ]:
npy_files = glob.glob(f"{patch_dir}/*.npy")
has_labels = os.path.exists(f"{WORKING}/data/processed/clusters_gps.csv")
print(f"Patches .npy : {len(npy_files)}")
print(f"Labels : {'OK' if has_labels else 'MANQUANT'}")
assert len(npy_files) > 0, "Aucun patch trouve"
assert has_labels, "clusters_gps.csv manquant"
print("Pret pour l'entrainement.")

## 7. Lancer l'entrainement

Activer le GPU : Runtime -> Change runtime type -> T4 GPU

~2h d'entrainement. Les logs s'affichent en direct.

In [ ]:
os.chdir(WORKING)
!python entrainer_resnet.py

## 8. Copier les resultats vers Drive

In [ ]:
os.makedirs(f"{DRIVE_PATH}/outputs/tables", exist_ok=True)
os.makedirs(f"{DRIVE_PATH}/outputs/models", exist_ok=True)
os.makedirs(f"{DRIVE_PATH}/outputs/figures", exist_ok=True)

for f in glob.glob(f"{WORKING}/outputs/tables/*"):
    shutil.copy(f, f"{DRIVE_PATH}/outputs/tables/")
for f in glob.glob(f"{WORKING}/outputs/models/*"):
    shutil.copy(f, f"{DRIVE_PATH}/outputs/models/")
for f in glob.glob(f"{WORKING}/outputs/figures/*"):
    shutil.copy(f, f"{DRIVE_PATH}/outputs/figures/")

shutil.copy(f"{WORKING}/data/processed/cnn_predictions_cluster.csv",
            f"{DRIVE_PATH}/outputs/")

print("Resultats sauvegardes dans :")
print(f"  {DRIVE_PATH}/outputs/")

## 9. Telecharger le zip (optionnel)

In [ ]:
!zip -r /content/results.zip $WORKING/outputs/ $WORKING/data/processed/cnn_predictions_cluster.csv
from google.colab import files
files.download('/content/results.zip')